# **Section 3: Revision - Booleans, conditionals, loops and simulation - Part 2**

## **Sampling and testing a model**

Part 1 was about generating randomness. This part is
about what you can conclude from it.

Needs `nba.csv` in the same folder.

#### **Helpful Resource:**
- [Python Reference](https://ulwazi.wits.ac.za/courses/89081/pages/detailed-python-reference-sheet-python-cheat-sheet-2?module_item_id=1200407)

**Recommended Readings:**
- **Chapters 10 to 11 of the textbook**

**Where this is used.** Foundational, as Part 1. `sample` returns in Section 5's
bootstrap and in the train/test split of Lab 7 and Project 2.


In [ ]:
# Run this cell first
%pip install -q datascience ipywidgets

try:
    import pyodide_http
    pyodide_http.patch_all()
except ImportError:
    pass

from datascience import *
import numpy as np
%matplotlib inline
import matplotlib.pyplot as plots
plots.style.use('fivethirtyeight')
import warnings
warnings.simplefilter('ignore', FutureWarning)

---

## **Contents**

1. [Population and sample, parameter and statistic](#1)
2. [Three ways to take a sample](#2)
3. [`.sample()`](#3)
4. [Empirical distributions and the law of averages](#4)
5. [The distribution of a statistic](#5)
6. [`sample_proportions`](#6)
7. [Testing a model](#7)
8. [When two categories are not enough: TVD](#8)
9. [**>>Quick questions<<**](#9)
10. [**>>Self-check<<**](#10)
11. [Quick reference](#11)


---

<a id='1'></a>
## **1. Population and sample, parameter and statistic**

Four words that get used loosely in ordinary speech and precisely here.

| Term | Means |
|---|---|
| **Population** | every individual you care about |
| **Sample** | the subset you actually measured |
| **Parametre** | a number describing the **population**, usually unknown |
| **Statistic** | a number computed from the **sample**, known, but varies |

The whole of Sections 3 to 5 is about one question: **how far is the statistic
likely to be from the parameter?**


In [ ]:
full_data = Table.read_table('nba.csv')
print(full_data.num_rows, 'players')
full_data.select('Player', 'Age', 'Salary').show(3)

In [ ]:
# A parameter: the average age of ALL these players
parameter = np.average(full_data.column('Age'))
parameter

In [ ]:
# A statistic: the average age of a sample of 20.
# Run this a few times -- it changes.
sample = full_data.sample(20, with_replacement=False)
np.average(sample.column('Age'))

The parameter is one fixed number. The statistic moves every time you take a new
sample. That variation is not a mistake. It is the thing being studied.


---

<a id='2'></a>
## **2. Three ways to take a sample**

Only one of them supports the reasoning in this course.

**Deterministic**, chosen by a rule with no chance involved. Every fifth row,
or all the rows matching some condition.

**Convenience**, whatever was easiest to reach. The first few rows.

**Random**, every individual has a known chance of being chosen.


In [ ]:
# Deterministic: every 50th player
full_data.take(np.arange(0, full_data.num_rows, 50)).select('Player', 'Age')

In [ ]:
# Convenience: the first five rows.
# This table is sorted by salary, so these are the five highest-paid players --
# not remotely typical.
full_data.take(np.arange(5)).select('Player', 'Salary')

In [ ]:
# Random: five rows, each equally likely
full_data.sample(5, with_replacement=False).select('Player', 'Age', 'Salary')

> **Why it matters.** The convenience sample above is not merely less accurate, 
> it is systematically wrong, and taking more of it would not help. A biased
> sample stays biased no matter how large it gets. Only random sampling lets you
> say anything about the population.


---

<a id='3'></a>
## **3. `.sample()`**

```
tbl.sample(k)                          # k rows, WITH replacement (the default)
tbl.sample(k, with_replacement=False)  # k rows, without replacement
tbl.sample()                           # as many rows as the table has
```

**With replacement** means the same row can be drawn more than once. That is the
default, and it catches people out.


In [ ]:
small = Table().with_columns('Letter', make_array('a', 'b', 'c'))

# With replacement -- repeats are possible
small.sample(5).column('Letter')

In [ ]:
# Without replacement -- you cannot ask for more rows than exist
small.sample(3, with_replacement=False).column('Letter')

---

<a id='4'></a>
## **4. Empirical distributions and the law of averages**

A **probability distribution** says what should happen in theory. An
**empirical distribution** is what actually happened when you tried it.

The **law of averages**: as the number of trials grows, the empirical
distribution gets closer to the theoretical one.


In [ ]:
die = Table().with_columns('Face', np.arange(1, 7))
die_bins = np.arange(0.5, 7.5, 1)

die.hist('Face', bins=die_bins)
plots.title('The theoretical distribution: every face equally likely')
plots.show()

In [ ]:
def roll_and_show(n):
    """Rolls a die n times and draws the empirical distribution."""
    rolls = die.sample(n)
    rolls.hist('Face', bins=die_bins)
    plots.title('Empirical distribution, ' + str(n) + ' rolls')
    plots.show()

roll_and_show(10)

In [ ]:
roll_and_show(1000)

Ten rolls look nothing like a flat distribution. A thousand rolls nearly do.
Nothing about the die changed, only the number of trials.


---

<a id='5'></a>
## **5. The distribution of a statistic**

Section 4 varied the number of rolls. This section varies something else: it
computes the **same statistic** on **many different samples**, and looks at how
the statistic itself is distributed.

This is the central idea of the course, and it has the same four-part shape as
any simulation.


In [ ]:
def sample_median_age(sample_size):
    """One trial: take a sample and return its median age."""
    sample = full_data.sample(sample_size, with_replacement=False)
    return percentile(50, sample.column('Age'))

sample_median_age(50)

In [ ]:
medians = make_array()

for i in np.arange(2000):
    medians = np.append(medians, sample_median_age(50))

Table().with_columns('Sample median age', medians).hist()
plots.title('2000 sample medians, each from a sample of 50')
plots.show()

print('population median:', percentile(50, full_data.column('Age')))

The histogram shows where the statistic tends to land. It clusters around the
population median, which is what makes a sample useful, but individual samples
can be some way off.

Try changing 50 to 200 and re-running. The histogram narrows.


---

<a id='6'></a>
## **6. `sample_proportions`**

```
sample_proportions(sample_size, distribution)
```

Draws a sample of the given size from a categorical distribution and returns the
**proportion** falling in each category. The input must be proportions that add
to 1.

It skips the individual draws entirely, which makes it fast, and means you
cannot ask about the order or identity of individuals.


In [ ]:
# A population that is 26% one group, 74% another
population = make_array(0.26, 0.74)

# One sample of 100
sample_proportions(100, population)

In [ ]:
# The proportions always add to 1
sum(sample_proportions(100, population))

> **Common mistake.** Passing counts instead of proportions. `make_array(26, 74)`
> does not add to 1, and the result will be wrong or will error. Divide by the
> total first.


---

<a id='7'></a>
## **7. Testing a model**

The framework, in four steps. It is the same every time.

| Step | What you do |
|---|---|
| 1 | State the **model**, a chance process that could have produced the data |
| 2 | Choose a **statistic** that would be large when the model is wrong |
| 3 | **Simulate** the statistic many times under the model |
| 4 | Compare the **observed** statistic to the simulated ones |

If the observed value sits far out in the tail, the model looks like a poor
explanation of what actually happened.

### **A worked example**

A panel of 100 people is drawn from a population that is 26% eligible in some
category. The panel contained 8. Is 8 surprising?


In [ ]:
# 1. The model: each panellist drawn at random, 26% chance
eligible_population = make_array(0.26, 0.74)

# 2. The statistic: the number in the panel from that group
def one_panel():
    """One trial under the model: returns the count out of 100."""
    return sample_proportions(100, eligible_population).item(0) * 100

one_panel()

In [ ]:
# 3. Simulate
panels = make_array()
for i in np.arange(10000):
    panels = np.append(panels, one_panel())

Table().with_columns('Count in panel', panels).hist(bins=np.arange(5.5, 46.5, 1))
plots.title('If the model is true')
plots.show()

In [ ]:
# 4. Compare
observed = 8
print('observed:', observed)
print('proportion of simulations at or below 8:', np.average(panels <= observed))

Under the model, a panel of 100 would typically contain around 26 people from
that group. In 10,000 simulated panels, **not one** came out as low as 8.

Be careful how you say what follows. This does not prove the model is false, and
it says nothing about why the panels looked as they did. What it says is that
the data would be very surprising if the model were true, which is as far as
this method goes.


---

<a id='8'></a>
## **8. When two categories are not enough: TVD**

With two categories, the statistic can be a simple difference. With more than
two, you need one number summarising how far apart two whole distributions are.

**Total variation distance** is that number:

$$\text{TVD} = \frac{1}{2}\sum \left| \text{observed} - \text{expected} \right|$$

Add up the absolute differences across every category, then halve it. The
halving is because every excess in one category is matched by a shortfall in
another, so without it you would count each gap twice.


In [ ]:
jury = Table().with_columns(
    'Ethnicity', make_array('Asian/PI', 'Black', 'Latino', 'White', 'Other'),
    'Eligible',  make_array(0.15, 0.18, 0.12, 0.54, 0.01),
    'Panels',    make_array(0.26, 0.08, 0.08, 0.54, 0.04))
jury

In [ ]:
jury.barh('Ethnicity')
plots.title('Eligible population against actual panels')
plots.show()

In [ ]:
def tvd(dist1, dist2):
    """Total variation distance between two distributions."""
    return sum(abs(dist1 - dist2)) / 2

observed_tvd = tvd(jury.column('Panels'), jury.column('Eligible'))
observed_tvd

Now the same four steps. The model says panels are drawn at random from the
eligible population; the statistic is the TVD.


In [ ]:
model = jury.column('Eligible')

def simulated_tvd():
    """One trial: TVD between a random panel of 1453 and the model."""
    return tvd(sample_proportions(1453, model), model)

tvds = make_array()
for i in np.arange(5000):
    tvds = np.append(tvds, simulated_tvd())

Table().with_columns('TVD', tvds).hist(bins=np.arange(0, 0.2, 0.005))
plots.title('Simulated TVDs if the model is true')
plots.show()

print('observed TVD:', round(observed_tvd, 4))
print('largest simulated TVD:', round(max(tvds), 4))

The observed TVD is far larger than anything the simulation produced in 5000
trials. Whatever generated those panels, the result does not look like random selection
from the eligible population.

Note what the plot does **not** tell you: why. A statistical test can say a
model fits badly. It cannot say what the right model is.


---
<a id = '9'></a>
## **9. Quick questions**

Five of them. Set `my_answer` to a letter and run the cell.
A wrong answer gets a nudge so you can try again; a right one gets the reason.

These come before the written questions below on purpose: they are quicker,
and they check the things people most often get wrong.

In [ ]:
# Run this once. mcq.py must be in the same folder as this notebook.
from mcq import check_answer, show_answer

**M1.** A table has 500 rows. Does `tbl.sample(500)` give you the same table back?

**a)** yes, all 500 rows in a new order  
**b)** no, it draws with replacement, so there are repeats  
**c)** yes, exactly the same table  
**d)** no, it gives 500 random rows without repeats  

In [ ]:
my_answer = '?'          # a, b, c or d
check_answer('w3b_m1', my_answer)

**M2.** Why does `sample_proportions(200, make_array(50, 30, 20))` fail?

**a)** the sample size is too small  
**b)** you need a table, not an array  
**c)** the second argument must be proportions summing to 1  
**d)** the array must have two entries  

In [ ]:
my_answer = '?'          # a, b, c or d
check_answer('w3b_m2', my_answer)

**M3.** `sample_proportions(200, dist)` returns `array([0.52, 0.29, 0.19])`. How many fell in the first category?

**a)** about 104  
**b)** 0.52  
**c)** 52  
**d)** you cannot tell  

In [ ]:
my_answer = '?'          # a, b, c or d
check_answer('w3b_m3', my_answer)

**M4.** A convenience sample is biased. What does taking a much bigger one do?

**a)** removes the bias  
**b)** halves the bias  
**c)** makes the bias worse  
**d)** nothing to the bias; size does not fix it  

In [ ]:
my_answer = '?'          # a, b, c or d
check_answer('w3b_m4', my_answer)

**M5.** You get a p-value of 0.02. What does that number mean?

**a)** there is a 2% chance the model is true  
**b)** 2% of simulations under the model were at least as extreme as what you saw  
**c)** there is a 98% chance your result is real  
**d)** the model is wrong 2% of the time  

In [ ]:
my_answer = '?'          # a, b, c or d
check_answer('w3b_m5', my_answer)

---
<a id = '10'></a>
## **10. Self-check**

Answer these without scrolling back. Reveal each answer only after you have committed to one.

**Q1.** A table has 500 rows and you call `tbl.sample(500)`. Do you get the same table back?

<details>
<summary><strong>Answer</strong></summary>

No. <code>sample</code> draws <strong>with replacement</strong> by default, so some rows appear more than once and others not at all. For a shuffle of the original rows use <code>tbl.sample(with_replacement=False)</code>.

</details>

**Q2.** What is the difference between a parameter and a statistic?

<details>
<summary><strong>Answer</strong></summary>

A <strong>parameter</strong> is a number about the whole population, fixed but usually unknown. A <strong>statistic</strong> is computed from a sample and changes from sample to sample. Inference is using the second to say something about the first.

</details>

**Q3.** `sample_proportions(200, [50, 30, 20])` fails. Why?

<details>
<summary><strong>Answer</strong></summary>

The second argument must be <strong>proportions that sum to 1</strong>, not counts. Written correctly: <code>sample_proportions(200, make_array(0.5, 0.3, 0.2))</code>.

</details>

**Q4.** `sample_proportions(200, dist)` returns `array([0.52, 0.29, 0.19])`. How many of the 200 fell in the first category?

<details>
<summary><strong>Answer</strong></summary>

About <strong>104</strong>. The output is proportions, not counts, so multiply by the sample size: <code>0.52 * 200</code>. Forgetting this is the most common error with this function.

</details>

**Q5.** Why use the total variation distance rather than a single difference when a model has more than two categories?

<details>
<summary><strong>Answer</strong></summary>

A single difference compares only one category. The TVD combines every category into one number: absolute differences, added, then halved. That gives one statistic you can simulate a distribution for.

</details>

**Q6.** `tbl.take(np.arange(0, 1000, 10))` gives 100 rows. Is that a random sample?

<details>
<summary><strong>Answer</strong></summary>

No, it is <strong>deterministic</strong>: every tenth row, chosen by position. Run it twice and you get the same rows. Whether it is representative depends entirely on how the table was ordered.

</details>

---

<a id='11'></a>
## **11. Quick reference**

### **Sampling**

| Call | Gives |
|---|---|
| `tbl.sample(k)` | k rows, **with** replacement |
| `tbl.sample(k, with_replacement=False)` | k rows, without replacement |
| `tbl.take(np.arange(0, n, step))` | a deterministic sample |
| `np.random.choice(array, k)` | k elements from an array |
| `sample_proportions(n, dist)` | the proportion in each category |

### **The four steps of a test**

| Step | Code shape |
|---|---|
| 1 Model | an array of proportions adding to 1 |
| 2 Statistic | a function returning one number per trial |
| 3 Simulate | `for i in np.arange(n): results = np.append(results, trial())` |
| 4 Compare | `np.average(results >= observed)` |

### **Vocabulary**

| Term | Is |
|---|---|
| Population | everyone you care about |
| Sample | who you measured |
| Parametre | a number about the population, fixed, usually unknown |
| Statistic | a number from the sample, known, varies |
| Empirical distribution | what actually happened |
| Probability distribution | what should happen in theory |

### **Things that catch people out**

| | |
|---|---|
| `.sample(k)` | with replacement by default |
| `sample_proportions` input | proportions, not counts; must sum to 1 |
| `sample_proportions` output | proportions, not counts, multiply by n |
| A larger biased sample | still biased; size does not fix bias |
| TVD | halve the sum, or you double-count |
| A small p-value | says the data is surprising under the model, not that the model is false |

---

### **Where this comes from in the textbook**

- [Chapter 10, Sampling and Empirical Distributions](https://inferentialthinking.com/chapters/10/sampling-and-empirical-distributions/)
- [Chapter 10.1, Empirical distributions](https://inferentialthinking.com/chapters/10/1/empirical-distributions/)
- [Chapter 10.3, Empirical distribution of a statistic](https://inferentialthinking.com/chapters/10/3/empirical-distribution-of-a-statistic/)
- [Chapter 11, Testing Hypotheses](https://inferentialthinking.com/chapters/11/testing-hypotheses/)
- [Chapter 11.2, Multiple categories](https://inferentialthinking.com/chapters/11/2/multiple-categories/)
